# 03-fix. Fallback Description 154건 재시도

503 에러로 실패한 154건의 Discriminative Description을 재생성한다.
기존 체크포인트에서 fallback 항목만 제거하고, 03 노트북의 전체 생성 셀을 재실행하면 미처리 건만 자동으로 처리된다.

## 0. 환경 설정

In [1]:
# === 환경 설정 ===
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/5stone_experiment/1_clustering_test')

import json
from pathlib import Path

CKPT_DIR = Path('checkpoints/descriptions')
print(f'작업 디렉토리: {os.getcwd()}')

Mounted at /content/drive
작업 디렉토리: /content/drive/MyDrive/5stone_experiment/1_clustering_test


## 1. 체크포인트에서 fallback 항목 제거
기존 배치 파일에서 `is_fallback=True`인 항목을 제거하여, 재실행 시 해당 cluster_id가 미처리로 인식되도록 한다.

In [2]:
# === Fallback 제거 ===

def remove_fallbacks_from_checkpoints():
    total_before = 0
    total_after = 0
    removed = 0

    for f in sorted(CKPT_DIR.glob('desc_batch_*.json')):
        with open(f, 'r', encoding='utf-8') as fp:
            batch = json.load(fp)

        total_before += len(batch)
        cleaned = [item for item in batch if not item.get('is_fallback', False)]
        batch_removed = len(batch) - len(cleaned)
        removed += batch_removed
        total_after += len(cleaned)

        if batch_removed > 0:
            if cleaned:
                with open(f, 'w', encoding='utf-8') as fp:
                    json.dump(cleaned, fp, ensure_ascii=False, indent=2)
                print(f'  {f.name}: {len(batch)} → {len(cleaned)} ({batch_removed}건 제거)')
            else:
                f.unlink()
                print(f'  {f.name}: 전체 삭제 (모두 fallback)')

    print(f'\n총 {total_before} → {total_after} ({removed}건 fallback 제거)')
    print(f'재실행 시 {removed}건이 미처리로 인식됩니다.')


remove_fallbacks_from_checkpoints()

  desc_batch_0000.json: 100 → 76 (24건 제거)
  desc_batch_0001.json: 100 → 79 (21건 제거)
  desc_batch_0002.json: 100 → 76 (24건 제거)
  desc_batch_0003.json: 100 → 85 (15건 제거)
  desc_batch_0004.json: 100 → 81 (19건 제거)
  desc_batch_0005.json: 100 → 87 (13건 제거)
  desc_batch_0006.json: 100 → 83 (17건 제거)
  desc_batch_0007.json: 100 → 90 (10건 제거)
  desc_batch_0008.json: 100 → 89 (11건 제거)

총 907 → 753 (154건 fallback 제거)
재실행 시 154건이 미처리로 인식됩니다.


## 2. 재실행 안내
이제 `03_discriminative_descriptions.ipynb`를 열어서 **§4-2 전체 생성 실행 셀**만 재실행하면 됩니다.
재개 로직이 이미 처리된 cluster_id를 건너뛰고, 제거된 154건만 새로 생성합니다.

재실행 후 §5(결과 검증)과 §6(최종 저장)도 다시 실행하여 `intent_descriptions.parquet`를 업데이트하세요.

In [3]:
# === 현재 상태 확인 ===

def check_current_state():
    processed = set()
    for f in sorted(CKPT_DIR.glob('desc_batch_*.json')):
        with open(f, 'r', encoding='utf-8') as fp:
            batch = json.load(fp)
        for item in batch:
            processed.add(item['cluster_id'])

    all_ids = set(range(907))
    remaining = all_ids - processed

    print(f'처리 완료: {len(processed)}')
    print(f'미처리 (재시도 대상): {len(remaining)}')
    if remaining:
        print(f'  예시 ID: {sorted(remaining)[:10]}...')


check_current_state()

처리 완료: 753
미처리 (재시도 대상): 154
  예시 ID: [6, 10, 22, 26, 29, 34, 35, 38, 49, 56]...
